# Interactive data exploration

source: https://data.oaklandca.gov/Infrastructure/Service-requests-received-by-the-Oakland-Call-Cent/quth-gb8e/about_data

First, setup dependencies and fetch data

In [ ]:
# install dependencies (esp if we are on colab)
%pip install pandas 'geopandas[all]' 'tqdm[notebook]' seaborn

In [ ]:
import pandas as pd

We are going to download the data from the Oakland data API.

It's about 300 MB (today)

If you don't want to download that much (takes about a minute on campus wifi today),
set `limit` to a smaller number, e.g. `10_000`.
It's about 2.5-3MB per 100 rows.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

data_path = Path("OAK_311.csv")
# reduce limit to get a smaller dataset
limit = 2_000_000
data_url = f"https://data.oaklandca.gov/resource/quth-gb8e.csv?$limit={limit}"

from tqdm.notebook import tqdm
import requests

def download(url: str, path: str | Path):
    """Download a file, but only if we don't have it already"""
    path = Path(path)
    if path.exists():
        print(f"Already have {path}")
    else:
        print(f"Downloading {url} to {path}")
        with tqdm(unit="B", unit_scale=True) as progress, path.open("wb") as f:
            r = requests.get(url, stream=True)
            r.raise_for_status()
            if 'Content-Length' in r.headers:
                progress.total = int(r.headers['Content-Length'])
            for chunk in r.iter_content(10_000):
                progress.update(len(chunk))
                f.write(chunk)

download(data_url, data_path)

Read the data!

In [ ]:
df = pd.read_csv(data_path)

Now, what data do we have?

In [ ]:
len(df)

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.head()

Sometimes datasets have a natural 'index' column.
It looks like requestid might be it

In [ ]:
df.requestid

In [ ]:
df = df.set_index("requestid")
df.head()

We can check what kind of content a few columns have

In [ ]:
df.councildistrict.value_counts()

In [ ]:
df.city.value_counts()

In [ ]:
df.state.value_counts()

In [ ]:
df.zipcode.value_counts()

It can be easier to navigate a wide table if we drop the columns we don't care about

In [ ]:
df = df.drop(["councildistrict", "city", "state"], axis=1)

In [ ]:
df.head()

What kinds of events are here?

In [ ]:
df.reqcategory.value_counts()

What is 'OTHER'?

In [ ]:
df[df.reqcategory == "OTHER"]

Looks like description has more detail.
What are the descriptions of these 'OTHER' events?

In [ ]:
df[df.reqcategory == "OTHER"].description.value_counts()[:20]

Seems like there are some miscategorization.
This dataset could use some cleaning!

What about referredto?

In [ ]:
df.referredto.value_counts()[:20]

We can also check which categories tend to get referred

In [ ]:
df[df.referredto.notna()].reqcategory.value_counts()

And who gets referred from a given category:

In [ ]:
df[df.reqcategory=="FIRE"].referredto.value_counts()

Or the reverse: which events gets referred to a given agency:

In [ ]:
df[df.referredto.notna() & df.referredto.str.contains("PG&E")].reqcategory.value_counts()

In [ ]:
df[df.referredto.notna() & df.referredto.str.contains("PG&E")].description.value_counts()

In [ ]:
df.description.value_counts()[:30]

## Exploring dates


In [ ]:
df.columns

In [ ]:
df.datetimeinit.iloc[0], df.datetimeinit.iloc[-1]

In [ ]:
df.datetimeinit.min(), df.datetimeinit.max()

Uh oh, these date strings are inconsistent and not sorted!

Let's convert them to actual dates first

In [ ]:
df["datetimeinit"] = pd.to_datetime(df.datetimeinit)
df["datetimeclosed"] = pd.to_datetime(df.datetimeclosed)

In [ ]:
df.datetimeinit.min(), df.datetimeinit.max()

In [ ]:
df.datetimeclosed.min(), df.datetimeclosed.max()

2006?! Let's investigate

In [ ]:
time_machine = df[df["datetimeclosed"] < df["datetimeinit"]]
time_machine.sort_values("datetimeclosed")

Let's assume any close date requiring time travel is invalid

In [ ]:
df.loc[df["datetimeclosed"] < df["datetimeinit"], "datetimeclosed"] = pd.NA
df.datetimeclosed.min(), df.datetimeclosed.max()

In [ ]:
df = df.sort_values("datetimeinit")
df.datetimeinit.iloc[0], df.datetimeinit.iloc[-1]

Now that we have real dates, when is our data from?

In [ ]:
counts_by_year = df.datetimeinit.dt.year.value_counts().sort_index()
counts_by_year

In [ ]:
import matplotlib.pyplot as plt

counts_by_year.plot(kind="barh")

plt.rc("axes", grid=False)
plt.rc('lines', linewidth=4)

In [ ]:
df.datetimeinit.dt.month.value_counts().sort_index().plot(kind="barh")

## Exploring columns

In [ ]:
df.head()

In [ ]:
df.source.value_counts()

In [ ]:
source_counts = df.groupby(df.datetimeinit.dt.year).source.value_counts()
source_counts

In [ ]:
source_counts.unstack()

In [ ]:
ax = source_counts.unstack().plot(kind="line")
ax.legend(bbox_to_anchor=(1, 1))

In [ ]:
import seaborn as sns
sns.heatmap(source_counts.unstack().fillna(0).transpose())

In [ ]:
ax = source_counts.unstack().plot(kind="line", )
source_counts.unstack().sum(axis=1).plot(
    ax=ax, kind="line", color="black", style="--", label="total",
)
ax.legend(bbox_to_anchor=(1, 1));

In [ ]:
import matplotlib.pyplot as plt

def reverse_legend(ax, **kwargs):
    """reverse the legend of an axes

    e.g. for streamgraph, stacked area where first is
    usually at the bottom.
    """
    if 'loc' not in kwargs:
        # outside grid, upper right
        kwargs.setdefault("bbox_to_anchor", (1, 1))
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], **kwargs)
    return ax
    
    
def streamgraph(df, *, ax=None, **kwargs):
    """Create a streamgraph"""
    if ax is None:
        ax = plt.gca()
    kwargs.setdefault("baseline", "weighted_wiggle")
    x = df.index
    y = df.fillna(0).values.T
    ax.stackplot(x, y, labels=df.columns, **kwargs)
    # reverse legend to align with stack plot order
    reverse_legend(ax)
    return ax
    
streamgraph(source_counts.unstack())

What about status?

In [ ]:
df.status.value_counts()

Are there any trends in status over time?

In [ ]:
counts = df.groupby(df.datetimeinit.dt.year).status.value_counts(normalize=True)
# streamgraph(counts.unstack(), baseline="sym")
ax = counts.unstack().plot(kind="area", linewidth=0)
reverse_legend(ax);


What about how long before a case is closed (for closed cases)

In [ ]:
df[df.datetimeclosed.notna()].status.value_counts()

In [ ]:
duration = df["datetimeclosed"] - df["datetimeinit"]
duration.dt.total_seconds().hist(grid=False)

In [ ]:
df["duration_days"] = duration.dt.total_seconds() / (24 * 3600)
df["duration_days"].value_counts()

In [ ]:
duration_bins = pd.cut(
    df["duration_days"], bins=[0, 1, 2, 7, 30, 365, 999999]
).cat.rename_categories(["1 day", "2 days", "1 week", "1 month", "1 year", "more"])
duration_bins

In [ ]:
duration_bins.value_counts().sort_index().plot(kind="barh")

In [ ]:
df["duration_days"].min(), df["duration_days"].max()

In [ ]:
df[df.duration_days > 3650]

In [ ]:
df[df.duration_days > 3650].description.value_counts()

Any trend in closing rate?

In [ ]:
df["duration_bins"] = duration_bins
counts = df.groupby(df.datetimeinit.dt.year).duration_bins.value_counts(normalize=True).unstack()
counts.plot()

In [ ]:
streamgraph(counts, baseline="sym")

In [ ]:
sns.heatmap(counts.transpose())
# counts.unstack().plot(kind="line")

In [ ]:
illdump = df[df.reqcategory=="ILLDUMP"]
counts = illdump.groupby(illdump.datetimeinit.dt.year).duration_bins.value_counts(normalize=True)
ax = counts.unstack().plot(kind="line")

In [ ]:
def plot_category_close_rate(category, normalize=True):
    # first filter to the selected category
    subset = df[df.reqcategory == category]
    # count by year (normalized)
    counts = subset.groupby(subset.datetimeinit.dt.year).duration_bins.value_counts(
        normalize=normalize
    ).unstack()
    # ax = streamgraph(counts.unstack(), baseline="sym")
    if normalize:
        ax = counts.plot(kind="area", linewidth=0)
        reverse_legend(ax, loc=0)
        ax.set_ylim(0, 1)
    else:
        ax = counts.plot(kind="line")
        
    
    ax.set_xlim(subset.datetimeinit.dt.year.min(), subset.datetimeinit.dt.year.max())

    # also draw the total number over time
    right_y = ax.twinx()
    subset.groupby(subset.datetimeinit.dt.year).status.count().plot(
        ax=right_y, color="black", style="--o", label="total"
    )
    right_y.legend(loc=0)
    right_y.grid(False)


plot_category_close_rate("ILLDUMP")

In [ ]:
plot_category_close_rate("PARKS")

Explore different categories

Check traffic, parks,  streetsw, meter repair, graffiti, parking, trees, homeless emt

In [ ]:
from ipywidgets import interact

interact(category=sorted(df.reqcategory.dropna().unique()))(plot_category_close_rate);

In [ ]:
df[df.reqcategory=="STREETSW"].description.value_counts()

## Location information

In [ ]:
import geopandas as gpd

In [ ]:
gpd.GeoSeries.explore?

In [ ]:
df.reqaddress

In [ ]:
df.reqaddress.iloc[0]

In [ ]:
reqaddress = df.reqaddress.iloc[0]
[float(s.strip()) for s in reqaddress[1:-1].split(",")]

In [ ]:
def parse_reqaddress(reqaddress):
    if pd.isna(reqaddress):
        return [pd.NA, pd.NA]
    return [float(s.strip()) for s in reqaddress[1:-1].split(",")]


latlon = df.reqaddress.apply(parse_reqaddress)
tuple(latlon.iloc[0])

In [ ]:
df[["srx", "sry"]]

In [ ]:
df[["reqaddress", "srx", "sry"]]

> | column | description |
> |---|---|
> | srx | Coordinates of the issue. Note that this is in NAD_1983_StatePlane_California_III_FIPS_0403_Feet, not latitude/longitude. |  
> | sry | Coordinates of the issue. Note that this is in NAD_1983_StatePlane_California_III_FIPS_0403_Feet, not latitude/longitude. |  

NAD_1983_StatePlane_California_III_FIPS_0403_Feet is `ESRI:102643`

In [ ]:
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["srx"], df["sry"], crs="ESRI:102643")
)

In [ ]:
# lose rows without location data
df = df.dropna(subset=["srx", "sry"])
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["srx"], df["sry"], crs="ESRI:102643")
)

In [ ]:
gdf.head()[["reqaddress", "srx", "sry", "geometry"]]

In [ ]:
gdf.to_crs(4326).head()[["reqaddress", "srx", "sry", "geometry"]]

In [ ]:
gdf[:100].explore()

![oakland](oakland.png)

[source (wikipedia)](https://en.wikipedia.org/wiki/Oakland,_California)



In [ ]:
gdf.head()[["reqaddress", "srx", "sry"]]

In [ ]:
geom = gpd.points_from_xy(df["srx"], df["sry"], crs=4326)
gdf = gpd.GeoDataFrame(df, geometry=geom)
gdf[-100:].explore()

In [ ]:
# don't need redundant coordinates anymore
gdf = gdf.drop(["reqaddress", "srx", "sry"], axis=1)

Select 2025 events

In [ ]:
gdf_2025 = gdf[gdf.datetimeinit.dt.year == 2025]
len(gdf_2025)

In [ ]:
gdf_2025[:10000].explore()

In [ ]:
gdf_2025[-1000:].explore(column="reqcategory")

Let's see even more data (all of it still too slow)

In [ ]:
gdf_2025[::5].explore(column="reqcategory")

Who is reporting an abandoned car from Oman?

Let's get some location info

- https://catalog.data.gov/dataset/500-cities-city-boundaries

In [ ]:
from IPython.display import display

if Path("500Cities_City_11082016").exists():
    %ls 500Cities_City_11082016
    shape = gpd.read_file("500Cities_City_11082016/CityBoundaries.shp").to_crs(gdf.crs)
    oakland = shape[(shape.NAME == "Oakland") & (shape.ST == "CA")]
    display(oakland.explore())

Also CA County boundaries:

- https://data.ca.gov/dataset/ca-geographic-boundaries

In [ ]:
county_data_path = Path("ca_counties.zip")
data_repo_raw_url = "https://github.com/Urban-Informatics-SP26/cp255_data/raw/refs/heads/main/w13"

county_data_url = f"{data_repo_raw_url}/{county_data_path.name}"
download(county_data_url, county_data_path)
counties = gpd.read_file(str(county_data_path))
counties.explore(column="NAME")

In [ ]:
counties.head()

Find our county

In [ ]:
counties = counties.to_crs(gdf_2025.crs)[["geometry", "NAME"]].rename({"NAME": "county_name"}, axis="columns")

In [ ]:
# alameda_county = counties.loc[counties.NAME == "Alameda", ["geometry", "NAMELSAD"]]
counties.loc[counties.county_name == "Alameda"].geometry.iloc[0]

What events were not in Alameda county?

Spatial join!

In [ ]:
# alameda_df = gpd.GeoDataFrame([alameda_county], columns=["geometry"])
gdf_2025 = gdf_2025.sjoin(counties, how="left")
outliers = gdf_2025[gdf_2025.county_name != "Alameda"]
outliers.explore(column="reqcategory")

In [ ]:
outliers

In [ ]:
gdf_2025 = gdf_2025[gdf_2025.county_name == "Alameda"]
gdf_2025[::5].explore(column="reqcategory")

In [ ]:
gdf_2025.reqcategory.value_counts()

In [ ]:
gdf_2025[gdf_2025.reqcategory == "TRAFFIC"].description.value_counts()

In [ ]:
subset = gdf_2025[gdf_2025.reqcategory == "TREES"]
subset.explore(column="description")

We can try to summarize in bins.
Why not use census tracts?

https://data.acgov.org/datasets/7b064a13a9234bfba97654007ccbf8e8/explore


In [ ]:
tract_boundaries_path = Path("Census_Tract_Boundaries_7506545346012929933.geojson")
tract_boundaries_url = f"{data_repo_raw_url}/{tract_boundaries_path.name}"
download(tract_boundaries_url, tract_boundaries_path)

tracts = gpd.read_file(tract_boundaries_path).to_crs(gdf.crs)
tracts

In [ ]:
tracts.explore(column="DIST_NAME", legend=False)

In [ ]:
tracts

In [ ]:
categories = tracts.sjoin(gdf_2025, how="left").groupby(["DISTRICT_ID", "reqcategory"]).reqcategory.count().unstack()
categories

Join with "inner" to drop tracts with no events

In [ ]:
tract_categories = tracts.set_index("DISTRICT_ID").join(categories, how="inner")
tract_categories


Check parks, dumping, trees, fire, meter repair, parking, drainage, abandoned auto

In [ ]:
from IPython.display import display
from ipywidgets import interact


@interact(category=sorted(gdf_2025.reqcategory.dropna().unique()))
def plot_category(category):
    display(tract_categories.explore(column=category, tooltip=[category, "DIST_NAME"]))

We also had more info by tract, such as population and median income,
in the dataset from Lab #6.

Census also has various other demographic data summarized by tract.

In [ ]:
tract_info_path = Path("alameda_tracts23.csv")
tract_info_url = f"{data_repo_raw_url}/{tract_info_path.name}"
download(tract_info_url, tract_info_path)

tract_info = pd.read_csv(tract_info_path)
tract_info

In [ ]:
tracts

In [ ]:
def tract_id(dist_name):
    _, tract_num = dist_name.split("#")
    if "." in tract_num:
        return int(tract_num.replace(".", ""))
    else:
        return int(tract_num) * 100


tracts["tract"] = tracts.DIST_NAME.apply(tract_id)

full_tracts = tracts.merge(tract_info, on="tract").drop(
    ["DIST_NAME", "NAME", "OBJECTID", "state", "county"],
    axis=1,
).set_index("DISTRICT_ID")
full_tracts.head()

In [ ]:
full_tracts.explore(column="population")

census info about that populous tract: https://censusreporter.org/profiles/14000US06001450751-census-tract-450751-alameda-ca/

Same information, this time divide each category count by the population in that tract

In [ ]:
full_tracts

In [ ]:
categories

In [ ]:
# categories = categories.set_index("tract")
per_capita_categories = categories.copy()
for column in categories.columns:
    per_capita_categories[column] = 1e3 * categories[column] / full_tracts.population
tracts_per_cap = full_tracts.join(per_capita_categories)
tracts_per_cap = tracts_per_cap.dropna(
    subset=gdf_2025.reqcategory.dropna().unique(), how="all"
)

tracts_per_cap

In [ ]:
tracts_per_cap.explore(column="TREES", tooltip=["population", "TREES", "tract"])

Try abandoned auto, dumping, parking, sewers, traffic, trees

In [ ]:
@interact(category=sorted(gdf_2025.reqcategory.dropna().unique()))
def plot_category_per_capita(category):
    display(tracts_per_cap.explore(column=category, tooltip=["tract", "population", category]))

Looks like almost everything happens by the bridge...

In [ ]:
tracts_per_cap.population.sort_values()

In [ ]:
tracts_per_cap = tracts_per_cap[tracts_per_cap.population > 1000]

Display it again, try abandoned auto, dumping, parking, sewers, traffic, trees

In [ ]:
plot_category_per_capita.widget

In [ ]:
full_tracts.explore(
    column="median_household_income", tooltip=["median_household_income", "population"]
)

In [ ]:
full_tracts.loc[full_tracts.median_household_income < 0, "median_household_income"]

In [ ]:
full_tracts.loc[full_tracts.median_household_income < 0, "median_household_income"] = pd.NA
full_tracts.explore(
    column="median_household_income", tooltip=["median_household_income", "population"]
)

Let's connect tracts to the raw data

In [ ]:
full_df = gdf.sjoin(full_tracts)

In [ ]:
full_df.head()

In [ ]:
closed = full_df[full_df.status == "CLOSED"]
closed = closed[["datetimeinit", "reqcategory", "median_household_income", "duration_days"]]
closed = closed[(closed.median_household_income >= 0) & (closed.duration_days > 0)]

closed.plot(x="median_household_income", y="duration_days", style=".")

In [ ]:
closed[closed.reqcategory == "HOMELESS EMT"].plot(
    x="median_household_income", y="duration_days", style="."
)

In [ ]:
import numpy as np

closed_homeless = closed[
    (closed.datetimeinit.dt.year == 2025) & (closed.reqcategory == "HOMELESS EMT")
]
ax = (
    closed.groupby(
        pd.cut(
            closed_homeless.median_household_income, bins=np.linspace(0, 300000, 31)
        ),
        observed=True,
    )
    .duration_days.median()
    .plot(kind="barh")
)
ax.set_xlabel("days")

In [ ]:
def plot_duration_by_income(closed, category, year=2025):
    subset = closed[(closed.datetimeinit.dt.year == year) & (closed.reqcategory == category)]
    ax = (
        closed.groupby(
            pd.cut(subset.median_household_income, bins=np.linspace(0, 300000, 31)),
            observed=True,
        )
        .duration_days.count()
        .plot(kind="barh", grid=False)
    )
    ax.set_xlabel("days")
    ax.set_title(f"{year} {category} close rate, by income")
plot_duration_by_income(closed, "TREES")

In [ ]:
plot_duration_by_income(closed, "ABANDONED AUTO")

### Exploring status

In [ ]:
df.status.unique()

In [ ]:
gdf_2025.status.value_counts()

In [ ]:
@interact(
    category=sorted(gdf_2025.reqcategory.dropna().unique()), status=sorted(gdf_2025.status.dropna().unique())
)
def plot_category_status(category, status):
    subset = gdf_2025[gdf_2025.reqcategory == category]

    def count_status(geom):
        return subset[subset.within(geom)].status.value_counts()

    statuses = tracts.geometry.apply(count_status)
    tract_status = full_tracts.join(statuses).dropna(subset=subset.status.dropna().unique(), how="all")
    if status not in tract_status.columns:
        print("No matching events")
        return

    display(
        tract_status.explore(
            column=status,
        )
    )

In [ ]:
df[df.status == "UNFUNDED"].groupby(
    df.datetimeinit.dt.year
).reqcategory.value_counts().unstack(1)


In [ ]:
df[(df.status == "UNFUNDED") & (df.reqcategory == "HE_CLEAN")].description.value_counts()

In [ ]:
df.referredto.value_counts()[:20]

In [ ]:
df[
    ~df.referredto.isna() & df.referredto.str.contains("hazard")
].referredto.value_counts()

In [ ]:
df[
    ~df.referredto.isna() & df.referredto.str.contains("hazard")
].description.value_counts()

In [ ]:
waste = gdf[gdf.description.str.contains("Waste")].copy()
waste["year"] = waste.datetimeinit.dt.year
waste.explore(column="year")